In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/post_cell_9.pickle

In [ ]:
%%cudf.pandas.profile
### cell 10 ###

# 1. Single‐pass grouping to get counts per country and type
counts = df_cell6.groupby(['first_country', 'type']).size().unstack().fillna(0)

# 2. Ensure both 'Movie' and 'TV Show' columns exist
for col in ['Movie', 'TV Show']:
    if col not in counts.columns:
        counts[col] = 0

# 3. Select only those two categories and compute the total per country
data_q2q3 = counts[['Movie', 'TV Show']]
data_q2q3['sum'] = data_q2q3['Movie'] + data_q2q3['TV Show']

# 4. Pick the top 11 countries by total titles
country_order = data_q2q3['sum'].nlargest(11).index
data_q2q3 = data_q2q3.loc[country_order]

# 5. Compute the ratio for each type and sort ascending by Movie ratio
data_q2q3_ratio = (
    data_q2q3[['Movie', 'TV Show']]
    .div(data_q2q3['sum'], axis=0)
    .sort_values('Movie', ascending=True)
)